# Timerseries Analysis

In [8]:
import librosa
import numpy as np
import pandas as pd
from tslearn.clustering import TimeSeriesKMeans
from tslearn.preprocessing import TimeSeriesScalerMeanVariance, TimeSeriesResampler
from tslearn.shapelets import LearningShapelets
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import glob

In [13]:
def extract_beat_sync_features(
    mp3_path: str,
    sr: int = 22050,
    trim_db: int = 25,
    hop_length: int = 512,
    n_mfcc: int = 13,
) -> dict:
    y, sr = librosa.load(mp3_path, sr=sr, mono=True)
    y, _ = librosa.effects.trim(y, top_db=trim_db)

    # normalize peak amplitude
    peak = np.max(np.abs(y)) + 1e-9
    y = y / peak

    duration = librosa.get_duration(y=y, sr=sr)

    # beat tracking
    onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop_length)
    tempo, beat_frames = librosa.beat.beat_track(
        onset_envelope=onset_env, sr=sr, hop_length=hop_length
    )
    if len(beat_frames) < 2:
        # fallback: if beat tracking fails, treat it like one "beat" at start
        beat_frames = np.array([0], dtype=int)

    beat_times = librosa.frames_to_time(
        beat_frames, sr=sr, hop_length=hop_length
    )

    # frame-level features
    mfcc = librosa.feature.mfcc(
        y=y, sr=sr, n_mfcc=n_mfcc, hop_length=hop_length
    )  # (n_mfcc, n_frames)

    chroma = librosa.feature.chroma_stft(
        y=y, sr=sr, hop_length=hop_length
    )  # (12, n_frames)

    # use the same onset envelope as a 1D rhythmic/dynamic feature
    onset = onset_env.reshape(1, -1)  # (1, n_frames)

    # beat-synchronous aggregation (median is robust)
    mfcc_bs = librosa.util.sync(mfcc, beat_frames, aggregate=np.median).T
    chroma_bs = librosa.util.sync(chroma, beat_frames, aggregate=np.median).T
    onset_bs = librosa.util.sync(onset, beat_frames, aggregate=np.median).T

    return {
        "mfcc_bs": mfcc_bs,
        "chroma_bs": chroma_bs,
        "onset_bs": onset_bs,
        "tempo": float(tempo),
        "beat_times": beat_times,
        "duration": float(duration),
    }

rows = []
features = []  # list of dicts returned by extract_beat_sync_features

for mp3_path in sorted(glob.glob("../dataset/fedez_fibra/*.mp3"))[:10]:
    out = extract_beat_sync_features(mp3_path)
    rows.append(
        {
            "artist": mp3_path.split(" ")[0],
            "path": str(mp3_path),
            "duration": out["duration"],
            "tempo": out["tempo"],
            "n_beats": int(out["mfcc_bs"].shape[0]),
        }
    )
    features.append(out)

meta = pd.DataFrame(rows)
meta.head()

/var/folders/xt/8hcc3d6j7l565z9k6z0w88jh0000gn/T/ipykernel_2483/3538853652.py:51: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  "tempo": float(tempo),


,artist,path,duration,tempo,n_beats
0,../dataset/fedez_fibra/ART07024718,../dataset/fedez_fibra/ART07024718 - TR106318.mp3,119.675646,89.102909,175
1,../dataset/fedez_fibra/ART07024718,../dataset/fedez_fibra/ART07024718 - TR113702.mp3,122.694240,123.046875,243
2,../dataset/fedez_fibra/ART07024718,../dataset/fedez_fibra/ART07024718 - TR116528.mp3,185.527438,123.046875,364
3,../dataset/fedez_fibra/ART07024718,../dataset/fedez_fibra/ART07024718 - TR124631.mp3,181.998005,92.285156,251
4,../dataset/fedez_fibra/ART07024718,../dataset/fedez_fibra/ART07024718 - TR128449.mp3,196.742676,129.199219,356


In [14]:
def summarize_song_features(out: dict) -> dict:
    mfcc = out["mfcc_bs"]          # (T, n_mfcc)
    chroma = out["chroma_bs"]      # (T, 12)
    onset = out["onset_bs"][:, 0]  # (T,)

    eps = 1e-9

    agg = {}
    agg["tempo"] = out["tempo"]
    agg["duration"] = out["duration"]
    agg["n_beats"] = float(mfcc.shape[0])

    # mean and std per coefficient
    mfcc_mean = mfcc.mean(axis=0)
    mfcc_std = mfcc.std(axis=0)
    for i, v in enumerate(mfcc_mean, start=1):
        agg[f"mfcc_mean_{i:02d}"] = float(v)
    for i, v in enumerate(mfcc_std, start=1):
        agg[f"mfcc_std_{i:02d}"] = float(v)

    agg["mfcc_var_mean"] = float(np.mean(mfcc_std**2))

    chroma_mean = chroma.mean(axis=0)  # average pitch-class energy
    chroma_std = chroma.std(axis=0)

    # normalize chroma_mean to a distribution for entropy-like features
    p = chroma_mean / (chroma_mean.sum() + eps)

    for k, v in enumerate(chroma_mean):
        agg[f"chroma_mean_{k:02d}"] = float(v)
    for k, v in enumerate(chroma_std):
        agg[f"chroma_std_{k:02d}"] = float(v)

    agg["chroma_entropy"] = float(-(p * np.log(p + eps)).sum())
    agg["chroma_max_frac"] = float(p.max())  # dominance of one pitch class

    # Onset strength summary stats (rhythm / dynamics proxy)
    agg["onset_mean"] = float(onset.mean())
    agg["onset_std"] = float(onset.std())
    agg["onset_p95"] = float(np.percentile(onset, 95))
    agg["onset_peakiness"] = float(
        np.percentile(onset, 95) / (onset.mean() + eps)
    )

    # simple "section change" proxy: how often onset changes sharply beat-to-beat
    diff = np.abs(np.diff(onset))
    agg["onset_diff_mean"] = float(diff.mean()) if diff.size else 0.0
    agg["onset_diff_p95"] = float(np.percentile(diff, 95)) if diff.size else 0.0

    return agg


# extract aggregated metadata features for all songs
agg_rows = []
for out in features:
    agg_rows.append(summarize_song_features(out))

X_meta = pd.DataFrame(agg_rows)
df = pd.concat([meta.reset_index(drop=True), X_meta.reset_index(drop=True)], axis=1)

# keep a clean numeric feature matrix for ML
feature_cols = X_meta.columns.tolist()
X = df[feature_cols].astype(float)

df.head()

,artist,path,duration,tempo,n_beats,tempo,duration,n_beats,mfcc_mean_01,mfcc_mean_02,...,chroma_std_10,chroma_std_11,chroma_entropy,chroma_max_frac,onset_mean,onset_std,onset_p95,onset_peakiness,onset_diff_mean,onset_diff_p95
0,../dataset/fedez_fibra/ART07024718,../dataset/fedez_fibra/ART07024718 - TR106318.mp3,119.675646,89.102909,175,89.102909,119.675646,175.0,-158.168900,103.729576,...,0.137189,0.132367,2.448570,0.128351,1.026717,0.360128,1.549208,1.508895,0.209277,0.556716
1,../dataset/fedez_fibra/ART07024718,../dataset/fedez_fibra/ART07024718 - TR113702.mp3,122.694240,123.046875,243,123.046875,122.694240,243.0,-51.307961,94.274467,...,0.191594,0.165256,2.473500,0.108482,1.307706,0.356755,1.850940,1.415410,0.290918,0.727805
2,../dataset/fedez_fibra/ART07024718,../dataset/fedez_fibra/ART07024718 - TR116528.mp3,185.527438,123.046875,364,123.046875,185.527438,364.0,-63.896343,82.414986,...,0.230848,0.194804,2.480602,0.093374,1.067506,0.340703,1.563847,1.464953,0.230953,0.711033
3,../dataset/fedez_fibra/ART07024718,../dataset/fedez_fibra/ART07024718 - TR124631.mp3,181.998005,92.285156,251,92.285156,181.998005,251.0,-12.326166,89.520187,...,0.287599,0.167687,2.468838,0.121238,0.928150,0.206752,1.219235,1.313618,0.156321,0.425156
4,../dataset/fedez_fibra/ART07024718,../dataset/fedez_fibra/ART07024718 - TR128449.mp3,196.742676,129.199219,356,129.199219,196.742676,356.0,-30.351585,69.020233,...,0.196356,0.249653,2.473906,0.107074,1.095254,0.333482,1.683079,1.536702,0.246248,0.824179


In [15]:
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))

# drop features with (near) zero variance
variances = X.var(axis=0)
keep = variances[variances > 1e-8].index
X = X[keep]
feature_cols = X.columns.tolist()

print("Songs:", len(df))
print("Features kept:", len(feature_cols))

Songs: 10
Features kept: 71


In [18]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled.shape

(10, 71)